# 추천시스템

- 추천 시스템 기초
    - 추천 시스템과 데이터 유형
    - 추천 시스템 분류
- 내용 기반 필터링
    - 사용자 프로필과 유사도
    - 장단점
- 협업 필터링
    - 메모리 기반 협업 필터링
    - 메모리 기반 협업 필터링의 한계
    - 모델 기반 협업 필터링
- VAE 기반 협업 필터링
    - MF의 한계와 VAE 구조
    - 목적 함수와 KL Divergence
- GNN 기반 사용자 선호 예측
    - GNN 구조와 프로세스
    - LightGCN
    - GNN 활용 사례 및 장단점
- Transformer 기반 추천 시스템
    - 기존 추천 시스템의 한계
    - Transformer과 Transformer 기반 주요 추천 시스템 모델
- 오늘날의 추천 시스템

## <span style="background-color:#fae0f0; color:#c0156d; padding:4px; border-radius:5px;"> Part 1. 추천 시스템 기초 </span>

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 1. 추천 시스템이란? </span>

- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 추천 시스템 </span> : 사용자의 행동 기록과 아이템 정보를 바탕으로 좋아할 만한 아이템을 예측해 골라 주는 시스템
    - ex. 넷플릭스 추천 콘텐츠, 유튜브 다음 영상, 이 상품을 본 고객이 함께 본 상품
- 필요한 이유
    - 고객 : 찾는 시간 감소, 몰랐던 취향 발견 (검색은 아는 것을 찾고, 추천은 모르는 것을 발견)
    - 기업 : 이용 시간, 구매 전환, 재방문 증가 / 비인기 아이템(롱테일) 노출

#### 1-1. 데이터의 종류
- **명시적 데이터** : 사용자가 선호를 직접 표현 (별점, 좋아요, 리뷰)
    - 장점 : 선호가 분명함
    - 단점 : 평가를 남기는 사람이 적음, 취향은 변함, 점수 기준이 사람마다 다름
- **암시적 데이터** : 행동에서 선호를 간접 추측 (조회, 클릭, 구매, 시청 시간, 검색)
    - 장점 : 따로 입력하지 않아도 대량으로 쌓임
    - 단점 : <span style="color:blue">행동이 곧 선호는 아님</span> (잘못 클릭, 선물 구매 / 행동이 없어도 아직 못 봤을 뿐일 수 있음)
- 실제 서비스는 암시적 데이터가 훨씬 많아 둘을 섞어 씀

#### 1-2. 추천 시스템의 분류
- 내용 기반 필터링 : <span style="color:blue">아이템 유사도</span>를 근거로
    - 사용자가 이용하거나 선택한 아이템과 비슷한 아이템을 추천하는 방식!
- 협업 필터링 : <span style="color:blue">**다수 사용자**의 행동 패턴</span>을 근거로
    - 사용자 기반 : 나와 취향이 비슷한 사용자가 좋아한 아이템 추천
    - 아이템 기반 : 내가 좋아한 아이템과 비슷한 아이템 추천
- 하이브리드 : 둘 이상을 조합해 서로의 단점을 보완
    - ex. 신작 영화는 평점이 없어 협업 필터링 불가 → 내용 기반으로 시작, 평점이 쌓이면 협업 필터링으로 전환
- 하이브리드 방식 (참고)
    - Weighted : 여러 모델의 결과를 점수로 합침
    - Mixed : 여러 알고리즘의 결과를 함께 보여줌
    - Switch : 상황에 따라 알고리즘을 선택 (위의 신작 영화 예시)
    - Feature Combination : 여러 feature를 모두 조합해 학습
    - Meta-Level : 앞 모델의 출력이 다음 모델의 입력


### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 2. 내용 기반 필터링 (Content-Based) </span>

- 흐름 : 아이템을 숫자로 표현 → 사용자 프로필 생성 → 유사도 계산 → 점수 높은 아이템 추천

#### 2-1. 사용자 프로필

- 직접 지정형 : 가입 시 선호 장르를 고르게 함
- 간접 지정형 : 과거 행동 이력에서 취향을 추론
- 프로필 획득 방식
    - ① 아이템 특성 벡터의 평균
        - ex. 어벤져스 [0.9, 0.1, 0.5, 0.0] (액션, 로맨스, SF, 공포), 인터스텔라 [0.3, 0.2, 0.9, 0.0], 노트북 [0.0, 0.9, 0.0, 0.1]
        - 평균 프로필 = [0.4, 0.4, 0.47, 0.03] → SF 선호가 가장 높음
        - 신규 영화 "듄" [0.4, 0.0, 0.9, 0.0]과 코사인 유사도 약 0.8 → 추천
    - ② 설문, 키워드, 평가로 따로 파악 (명시적: 좋아요, 평점, 댓글 / 암시적: 검색 로그, 구매, 북마크)
    - ③ 유저 평가를 정답 라벨로 쓰는 <span style="color:blue">분류 문제</span>로 해결 (입력: 아이템 특징, 정답: 유저 평가 → 결정 트리, 랜덤 포레스트, KNN)

#### 2-2. 유사도

- 아이템과 사용자를 같은 feature 기준의 벡터로 표현해야 비교 가능
- 아이템을 숫자로 : 속성(장르, 감독, 배우 등)을 feature로 두고 <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 원-핫 인코딩 </span> (해당하면 1, 아니면 0)
- 벡터 내적 : 같은 위치끼리 곱해서 모두 더함. 값이 클수록 유사 (0/1 벡터면 겹치는 feature 개수)
- 유사도 종류
    - 자카드 : 집합의 교집합 크기
    - 코사인 : 벡터 사이의 각도
    - 유클리디안 거리 : 벡터 사이의 거리 (작을수록 유사)
    - 피어슨 상관계수 : 선형 상관관계
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 코사인 유사도 </span> : $\cos(A,B) = \dfrac{A \cdot B}{\|A\|\|B\|}$
    - 길이는 무시하고 <span style="color:blue">방향만 비교</span> (ex. [1,1]과 [5,5]는 유사도 1)
    - <span style="color:green">거리처럼 쓰려면 코사인 거리 = 1 − 코사인 유사도</span>
- 내적 vs 코사인
    - 내적 : 길이까지 반영 → feature가 많은 아이템(장르가 많은 영화)이 유리
    - 코사인 : 길이를 무시 → feature 개수로 인한 쏠림은 줄지만 정보가 풍부하다는 점은 반영 못 함

#### 2-3. TF-IDF

- 단어 횟수만 세면 "그리고", "있다" 같은 흔한 단어가 중요해 보이는 문제
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> TF-IDF </span> : <span style="color:blue">특정 문서에는 자주 나오고 다른 문서에는 드문 단어</span>에 높은 값
    - TF : 문서 d에서 단어 t의 등장 횟수
    - DF : 단어 t가 등장한 문서 수
    - $IDF = \log\left(\dfrac{n}{1 + DF(t)}\right)$ ($n$: 전체 문서 수)
    - $TF\text{-}IDF = TF \times IDF$
- ex. 문서 3개 중 "귀엽다"는 2개에 등장 → IDF = log(3/3) = 0 → 중요도 낮음 / "사냥한다"는 1개에만 등장 → IDF = log(1.5) ≈ 0.176 → 상대적으로 중요
- 추천에 활용 : 줄거리를 TF-IDF 벡터로 변환 → 코사인 유사도로 비교 → 유사도 높은 영화 추천

#### 2-4. 장단점

- 장점
    - 속성만 있으면 신규 아이템도 추천 가능 (아이템 콜드 스타트 해결)
    - 다른 사용자의 데이터가 필요 없음
    - 추천 이유를 설명하기 쉬움
- 단점
    - feature로 담을 수 있는 정보에 한계 (속성이 부족하면 정확도 하락)
    - 비슷한 것만 계속 추천 (과도한 특화, Over-specialization)
    - 신규 사용자는 추천하기 어려움 (사용자 콜드 스타트)
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 콜드 스타트 </span> : 정보가 충분히 쌓이지 않아 추천하지 못하는 문제
    - 아이템 콜드 스타트 : 새 아이템에 평가가 없음 (내용 기반으로 해결 가능)
    - 사용자 콜드 스타트 : 새 사용자의 기록이 없음 (내용 기반으로도 어려움)

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 3. 협업 필터링 (Collaborative Filtering) </span>

- 가정 : <span style="color:blue">과거에 비슷한 선호를 가진 사용자들은 앞으로도 비슷한 선호를 가질 가능성이 높다</span>

#### 3-1. 메모리 기반 협업 필터링

- 저장된 평점을 그대로 꺼내 새 평점을 예측. 별도 학습 단계 없이 비슷한 "이웃"을 찾아 쓰므로 이웃 기반(Neighborhood-based) CF라고도 함
- 사용자 기반 : "당신과 비슷한 고객들이 이 상품도 구매했습니다"
    - 유사도 높은 TOP-N 사용자를 이웃으로 선정 → 이웃 평점을 유사도로 가중평균해 예측 → 예측 평점이 높은 아이템 추천
    - ex. A와 B가 어바웃 타임, 노팅힐에 모두 높은 점수 → B가 높게 평가한 라라랜드를 A에게 추천
    - 사용자 간 유사도 : 평점 데이터에서는 <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 피어슨 상관계수 </span>를 많이 씀
        - $sim(u,v) = \dfrac{\sum_i (r_{ui}-\bar r_u)(r_{vi}-\bar r_v)}{\sqrt{\sum_i (r_{ui}-\bar r_u)^2}\sqrt{\sum_i (r_{vi}-\bar r_v)^2}}$
    - 예측 평점 : $\hat r_{ui} = \bar r_u + \dfrac{\sum_v sim(u,v)(r_{vi}-\bar r_v)}{\sum_v |sim(u,v)|}$
        - 이웃이 자기 평균보다 얼마나 높게(낮게) 줬는지를 유사도만큼 반영해 내 평균에 더함
    - ex. sim(A,B) ≈ 0.97, sim(A,C) = −1, A 평균 3.33 → 라라랜드 예측 ≈ 3.96 → 추천
- 아이템 기반 : "이 상품을 선택한 고객들은 이 상품도 구매했습니다"
    - 속성이 아니라 <span style="color:blue">사용자들이 함께 좋아하고 싫어한 패턴</span>으로 아이템 유사도 판단
    - 아이템 관계는 상대적으로 덜 변해서 유사도를 미리 계산해 재사용하기 좋음
    - ex. 어바웃 타임을 높게 준 사람들이 노팅힐도 높게 줌 → 둘은 유사한 아이템

| 기준 | 사용자 기반 | 아이템 기반 |
|:---:|:---:|:---:|
| 정확성 | 아이템 수가 사용자보다 많을 때 유리 | 사용자 수가 아이템보다 많을 때 유리 |
| 안정성 | 아이템이 자주 바뀌고 사용자 유입이 적은 환경 | 아이템 목록이 잘 변하지 않는 환경 |
| 계산 속도 | 추천 때마다 비슷한 사용자를 새로 찾음 | 유사도를 미리 계산해 두고 재사용 → 빠름 |
| 다양성 | 의외성, 참신성이 강함 | 기존 아이템과 비슷한 추천이 많음 |
| 활용 예 | 뉴스, 소셜 미디어, 음악 | 아마존 등 사용자가 많은 대형 플랫폼 |

- 사용자 수가 많으면 아이템 기반이 유리한 이유 : 두 사용자가 함께 평가한 아이템은 적어도, 두 아이템을 함께 평가한 사용자는 많음. 아이템 유사도 계산은 가끔만 하면 됨
- 정답이 있는 것은 아니고 일반적인 경향

#### 3-2. 메모리 기반의 한계 3가지

- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 희소 행렬 문제 </span> : 사용자-아이템 행렬의 대부분이 비어 있음
    - ex. 넷플릭스 프라이즈 : 약 85억 칸 중 실제 평점 약 1억 개 → 약 99%가 비어 있음
    - 함께 평가한 아이템이 적어 유사도를 믿기 어렵고, <span style="color:green">공통 아이템이 1개뿐이어도 코사인 유사도가 1로 나올 수 있음</span>
    - 해결 : 모델 기반 CF (3-3)
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 콜드 스타트 </span> : 행동 기록에만 의존하므로 기록이 없으면 추천 불가
    - 신규 사용자 / 신규 아이템 (first-rater 문제)
    - 해결 : 가입 시 선호 장르 지정, 새 아이템엔 내용 기반 병행(하이브리드), 같은 카테고리 임베딩 평균으로 임시 표현, 인구통계 정보 활용
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 확장성 문제 </span> : 사용자와 아이템이 늘수록 계산량 증가
    - 해결 : KNN 협업 필터링 (유사도 높은 k명의 이웃만 사용, 이웃은 해당 아이템을 실제 평가한 사용자 중에서만 선택)
    - k가 너무 작으면 소수 의견에 흔들리고, 너무 크면 덜 비슷한 사용자가 섞여 예측이 흐려짐

#### 3-3. 모델 기반 협업 필터링

- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 모델 기반 CF </span> : 관측된 평점으로 모델을 학습해 패턴을 파라미터에 담고, 비어 있는 평점을 예측
    - 메모리 기반 : 추천 때마다 저장된 평점을 직접 비교
    - 모델 기반 : 평점 데이터를 먼저 압축된 모델로 요약해 두고 추천은 이 모델로만 수행
- 두 단계 : 학습 단계(미리, 새 데이터가 쌓이면 주기적 재학습) → 예측 단계(추천 시점에 빠르게)
- 등장 배경 : 메모리 기반은 평점 사이의 유사도에만 의존해 숨은 취향 구조를 못 잡고 희소성, 확장성에 취약
- 가장 대표적인 것이 잠재 요인 모델, 이를 구현하는 핵심 기법이 행렬 분해

**잠재 요인 (Latent Factor)**

- 평점에 직접 드러나지 않지만 사용자와 아이템의 관계를 결정짓는 숨은 특성 ("취향", "성향" 같은 개념을 숫자로 표현)
- Factor Model : 축의 의미를 사람이 미리 정함 (액션, 로맨스 등)
- Latent Factor Model : 축의 개수(k)만 정하고 <span style="color:blue">축의 의미는 학습으로 결정</span> → 사람이 해석하기 어려움
- 사용자와 아이템을 같은 잠재 공간의 벡터로 놓고, 거리나 각도로 유사도를 파악

**행렬 분해 (Matrix Factorization)**

- 행렬 : 행=사용자, 열=아이템, 칸=평점 (사용자 3명, 아이템 4개면 3×4)
- 행렬 곱 : (3×2)와 (2×4)를 곱하면 (3×4). 결과의 각 칸은 한 행과 한 열의 내적
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 행렬 분해 </span> : 큰 평점 행렬을 작은 행렬 두 개의 곱으로 쪼갬
    - $\hat R = P^T Q$
    - $P$ ($k \times m$) : 사용자 잠재 요인 행렬 / $Q$ ($k \times n$) : 아이템 잠재 요인 행렬 / $k$ : 잠재 요인 개수 (하이퍼파라미터)
- 예측 평점 : $\hat r_{ui} = p_u^T q_i = \sum_{s=1}^{k} p_{su} q_{si}$
    - ex. 사용자 p=(0.9, 0.2), 노팅힐 q=(0.8, 0.1) → 0.74 / 존 윅 q=(0.1, 0.9) → 0.27 → 노팅힐 추천
    - <span style="color:green">"로맨스", "액션"은 설명용 이름일 뿐, 실제 잠재 요인은 이렇게 깔끔히 해석되지 않는 경우가 대부분</span>
- 학습 과정
    - ① k 정하기 → ② 벡터를 임의 값으로 시작 → ③ 평점이 있는 칸만 예측하고 오차 계산 → ④ 오차가 줄도록 벡터 수정 → ⑤ 반복 → ⑥ 빈칸 예측
- 손실 함수 : $\sum_{\text{평점 있는 칸}} (\text{실제} - \text{예측})^2 + \lambda \times (\text{벡터 크기})^2$
    - 앞부분 : 오차 / 뒷부분 : 규제 (벡터가 지나치게 커지는 것을 막아 과적합 방지, λ가 클수록 강한 규제)
- 벡터를 고치는 방법
    - SGD : 평점을 하나씩 보며 오차가 줄어드는 방향으로 조금씩 수정
    - ALS : 사용자 벡터와 아이템 벡터를 번갈아 고정하며 나머지를 계산 (병렬 처리에 유리)
- 편향(Bias) : $\hat r_{ui} = \mu + b_i + b_u + p_u^T q_i$
    - $\mu$ 전체 평균, $b_i$ 아이템 편향, $b_u$ 사용자 편향
    - ex. 전체 평균 3.7, 타이타닉 +0.5, 깐깐한 Joe −0.3 → 기본 예측 3.9점
    - 편향이 경향을 먼저 설명하면 잠재 요인은 "이 사용자와 이 아이템이 얼마나 잘 맞는지"에 집중
- 행렬 분해 방식
    - SVD : 수학적 분해. 희소 행렬에는 직접 적용 불가(결측치 처리 필요)
    - 학습 기반 MF (SGD, ALS) : 관측된 평점만 학습해 희소성을 우회
    - NMF : 비음수 제약을 추가해 해석력 향상
    - PMF : 평점이 사용자·아이템 벡터 내적을 평균으로 하는 정규분포를 따른다고 보는 확률적 접근
- 메모리 기반 vs 모델 기반
    - 예측 : 이웃을 직접 찾음 vs 학습한 벡터를 내적 (빠름)
    - 콜드 스타트 : 둘 다 해결 어려움 (상호작용이 없으면 벡터를 학습할 수 없음)
    - 장점 : 설명하기 쉬움 vs 데이터 압축, 일반화에 유리
- <span style="color:blue">행렬 분해는 희소성과 확장성을 많이 해결했지만, 내적이라는 선형 계산으로 예측한다는 한계가 남음</span> → Part 2

## <span style="background-color:#fae0f0; color:#c0156d; padding:4px; border-radius:5px;"> Part 2. 딥러닝 기반 추천 시스템 </span>

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 1. VAE 기반 협업 필터링 </span>

#### 1-1. MF의 한계와 딥러닝의 등장

- ① 선형성 : 예측이 내적(요인별 곱의 합)이라 요인끼리 얽히는 효과를 표현하기 어려움
    - ex. 로맨스도 음악 영화도 좋지만 뮤지컬 로맨스는 별로인 사용자 → 내적에서는 둘 다 높은 영화가 오히려 최고점
    - 딥러닝의 해법 : 비선형 활성화 함수를 가진 신경망
- ② 불확실성 무시 : 사용자마다 벡터 하나만 학습(점 추정, Point Estimate)해서 기록 5개인 사용자와 5,000개인 사용자를 똑같이 확신함
    - 딥러닝의 해법 : 평균과 분산을 함께 출력
- 두 한계를 극복하려고 도입된 모델이 VAE

#### 1-2. 왜 AE가 아니라 VAE인가?

- 오토인코더(AE) : 입력을 작은 벡터로 압축했다가 복원하는 신경망 (입력 자체가 정답)
    - 참고 AutoRec : AE를 협업 필터링에 적용한 초기 모델. 은닉층 선형 RMSE 0.872 → sigmoid 0.831 (낮을수록 좋음) → 비선형성이 성능을 높임
- <span style="color:blue">AE도 사용자를 점 하나로 표현한다는 점은 MF와 같음</span>
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> VAE </span> : 잠재 벡터를 점이 아닌 <span style="color:blue">분포</span>로 표현하는 오토인코더
    - 평균 μ : 취향의 중심 위치 / 표준편차 σ : 그 위치를 얼마나 확신하지 못하는지(불확실성)
- VAE가 더 안정적인 이유
    - 학습 때마다 μ 근처에서 z를 무작위로 뽑아 디코더에 넣어, μ에서 조금 벗어나도 비슷한 추천이 나오도록 학습됨
    - KL 항이 분포를 표준 정규분포 쪽으로 당겨 기록 몇 개만으로 극단적인 위치로 가기 어려움
- AE는 단일 값, 불확실성 반영 안 함 / VAE는 μ와 σ를 가진 정규분포, 사용자별 σ로 불확실성 표현

#### 1-3. Mult-VAE의 구조

- 입력 x : 아이템 수만큼 긴 0/1 벡터 (평점은 4점 이상만 1)
- 인코더 : μ, σ 출력
- 잠재 벡터 z : μ, σ로 정해진 정규분포에서 샘플링
- 디코더 : 모든 아이템의 점수를 계산해 softmax로 확률로 변환 → 아직 상호작용하지 않은 아이템 중 확률 높은 순으로 추천

#### 1-4. 목적 함수와 KL Divergence

- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> ELBO </span> 최대화 : 두 목표를 동시에 최적화
    - $L = E_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) \| p(z))$
    - 복원 항 : z로 원래 입력을 최대한 비슷하게 복원 (AE의 역할)
    - KL 항 : 인코더의 잠재 분포 q(z|x)가 표준 정규분포 p(z)에서 크게 벗어나지 않도록 강제 → 과적합 방지, 잠재 공간을 연속적이고 부드럽게 유지
    - 비유 : 1번은 "기출 문제를 많이 풀어라", 2번은 "기본 개념에서 벗어나지 마라". 1번만 있으면 기출 답만 외우고, 2번만 있으면 모든 문제에 같은 답을 씀
- KL Divergence : 두 확률분포가 얼마나 다른지 나타내는 값
    - $D_{KL}(N(\mu,\sigma^2) \| N(0,1)) = \dfrac{1}{2}(\mu^2 + \sigma^2 - \ln\sigma^2 - 1)$
    - μ=0, σ=1 → 0 (벌점 없음) / μ=2, σ=1 → 2.0 (중심에서 멀음) / μ=0, σ=0.1 → 약 1.81 (너무 좁게 확신) / μ=0, σ=3 → 약 2.90 (너무 넓게 퍼짐)
    - <span style="color:blue">중심에서 멀어지거나, 지나치게 확신하거나, 지나치게 퍼지면 벌점이 커짐</span>
- AE vs VAE의 잠재 공간
    - AE : 복원만 하면 되므로 벡터가 흩어지고 사이의 빈 공간은 학습되지 않음. 기록이 적은 사용자는 우연한 기록(노이즈)으로 엉뚱한 곳에 놓이기 쉬움
    - VAE : N(0,1) 쪽으로 당겨져 중심 근처에 연속적으로 모임. 기록이 적은 사용자도 비슷한 사용자 근처에 안정적으로 놓임
- VAE-CF의 장점 : 불확실성 반영(확률적 잠재 분포), 희소한 데이터에서 안정적, 비선형 패턴 학습

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 2. GNN 기반 사용자 선호 예측 </span>

- CF, MF, VAE는 상호작용을 행렬의 칸 하나하나로 다룸
    - "A가 본 영화를 B도 봤고, B는 영화 Y도 봤다" 같은 <span style="color:blue">이어진 관계</span>는 개별 칸으로 활용 불가, 행동 순서도 고려하지 않음
- 사용자-아이템 상호작용은 본질적으로 그래프 : 나 → 영화 X → (X를 본) 다른 사용자 → 영화 Y

#### 2-1. 그래프

- 노드(Node) : 개체 (사용자, 아이템 등)
- 엣지(Edge) : 노드 사이의 관계나 상호작용. 방향을 가질 수 있고(팔로우), 가중치를 가질 수 있음(평점 5점, 구매 3회)
- 그래프를 쓰는 이유
    - "취향이 비슷하다", "함께 구매한다" 같은 추상적 관계를 노드-엣지-노드로 직접 저장해 학습
    - Non-Euclidean 공간 데이터를 표현하고 학습 가능
        - 이미지(격자)와 텍스트(시퀀스)는 이웃 구조가 고정되어 CNN, RNN 적용 가능
        - 그래프는 노드마다 이웃 수가 제각각이고 정해진 순서나 좌표가 없어 GNN이 필요
        - 그래프를 행렬로 펼쳐 기존 신경망에 넣으면 정보 손실이나 비효율 발생

#### 2-2. GNN의 구조와 프로세스

- 구조 : ① 그래프 생성(노드, 엣지, 초기 임베딩) → ② 메시지 패싱 → ③ 최종 출력
- 핵심 아이디어 : <span style="color:blue">내 이웃 노드(비슷한 사용자, 내가 본 아이템)의 정보를 나에게 전달, 요약</span> (Message Passing)
- 그래프 생성 예 : 사용자 3명 + 영화 3개 = 노드 6개, 평점 기록 5개 = 엣지 5개
    - 인접 행렬 : 노드 간 관계 (6×6, 연결되면 1)
    - 피처 행렬 : 노드 고유의 feature 정보 (관계 정보 제외)
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 메시지 패싱 </span> : 한 레이어는 메시지 생성 → 집계 → 갱신. K개 레이어면 K번 반복, 모든 노드가 한 번씩 타깃 노드가 됨
    - 메시지 생성 : 이웃 노드가 자신의 임베딩으로 메시지를 만듦 ($m_u^{(l)} = MSG^{(l)}(h_u^{(l-1)})$)
    - 집계 : 여러 이웃의 메시지를 하나로 합침 (sum, mean, max, attention-weighted sum)
    - 갱신 : 집계 결과와 자신의 이전 임베딩을 결합해 새 임베딩 계산 (신경망 레이어 W와 활성화 함수 σ 통과)
    - <span style="color:blue">l번 반복하면 l단계 떨어진 이웃의 정보까지 반영</span>
    - ex. U1이 타깃이면 영화 A, B의 메시지를 평균해 요약 → U1의 이전 임베딩과 결합

#### 2-3. 주요 GNN 아키텍처

- 차이를 결정하는 두 가지 : 메시지를 어떻게 만드는가, 어떻게 합치는가
- GCN : 그래프 위에서 Convolution. 이웃 임베딩을 연결 수(차수)로 정규화해 평균. 가장 기본적인 구조
- GRN : RNN + GNN 결합. 시간에 따라 변하는 그래프(동적 관계) 포착
- GAT : attention으로 이웃마다 중요도를 학습해 가중 평균. 중요한 이웃에 집중하지만 계산량 증가
- GraphSAGE : 이웃을 일부만 샘플링하고 이웃 정보를 더하지 않고 자기 정보와 concat. 대규모 그래프와 학습 때 없던 새 노드에도 적용 가능
- 비유 (카페 사장이 민수의 주문을 친구들 의견으로 예측)
    - 엣지(친구 관계)와 노드 특징(친구들의 주문 기록)은 고정, W(의견을 반영하는 판단 기준)는 학습으로 바뀜
    - GNN : 친구 의견을 똑같이 평균 / GAT : 취향이 비슷한 친구 의견에 더 큰 비중
- <span style="color:green">GNN은 그래프 데이터를 학습하는 신경망 전체를 가리키는 개념, GCN은 그중 합성곱으로 이웃 정보를 집계하는 구체적인 모델</span>

#### 2-4. LightGCN

- 추천에서는 GCN을 적용한 NGCF(2019)가 먼저 나왔고, 이를 단순화한 LightGCN(2020)이 뒤이어 등장
- NGCF는 레이어마다 가중치 행렬 W를 곱하고 비선형 활성화를 적용 → 풍부한 피처가 있을 때 유용
- 하지만 추천 그래프의 노드에는 <span style="color:blue">ID 임베딩만</span> 있어서, 이 두 연산이 성능에 도움이 안 되고 오히려 학습을 방해
- 제거한 것 : ① 특징 변환(W) ② 비선형 활성화 함수 (ReLU 등)
    - 활성화 함수를 제거한 이유 : 임베딩 전파 과정에서 이미 비선형성이 충분히 반영됨, 과적합 방지
- 정규화된 이웃 임베딩의 합 + 레이어 결합만으로 구성. NGCF보다 평균 약 16% 높은 성능 (He et al., SIGIR 2020)

#### 2-5. GNN 활용 사례

- Pinterest : 핀과 보드를 노드, '핀이 보드에 저장됨'을 엣지로 하는 그래프 (약 30억 노드, 180억 엣지). 전체 연산이 불가능해 GraphSAGE처럼 중요한 이웃만 샘플링해 학습
- 소셜 네트워크(친구 추천), 화학·생명과학(분자 구조, 약물 상호작용), 금융·보안(이상 거래 탐지), 교통(소요 시간 예측), 프로그램 분석(코드 자동 제안)

#### 2-6. 장단점

- 장점
    - 고차원 협업 시그널 포착 : 이웃과 연관 아이템의 정보를 반복해서 반영
    - 상호작용, 소셜 관계, 지식 그래프, 사용자 속성을 하나의 그래프로 학습
    - 상호작용 없는 신규 아이템도 속성 노드나 이웃으로 임베딩을 얻어 콜드 스타트 완화
- 단점
    - <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 과평탄화(Over-smoothing) </span> : 레이어가 많아지면 모든 노드의 임베딩이 비슷한 값으로 수렴
    - <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 과압축(Oversquashing) </span> : 기하급수적으로 늘어나는 이웃 정보를 고정 크기 벡터에 담으며 먼 노드의 정보가 손실
    - 계산 복잡도 증가 (수십억 노드, 엣지)
    - <span style="color:blue">순서 반영 어려움</span> : 신발→바지→셔츠와 셔츠→바지→신발이 같은 그래프가 됨

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 3. Transformer 기반 추천 시스템 </span>

#### 3-1. 기존 추천의 순차적 한계

- CF : 순서 무시, 문맥 부족('방금'과 '1년 전' 구매를 구분 못 함), '다음' 행동보다 '일반적인' 선호를 예측
- GNN : 정보가 레이어를 거치며 희석되고(과압축), 레이어 수만큼의 단계까지만 볼 수 있어 장기 의존성 포착이 어려움
- <span style="color:blue">CF는 '누가 무엇을 좋아하는가', GNN은 '무엇이 무엇과 연결되었는가'를 잘 포착하지만, 둘 다 '어떤 순서로, 지금 어떤 의도로' 행동하는지는 못 봄</span>
- 대안 : 시퀀스 전체를 한 번에 보며 지금의 맥락에 중요한 과거 행동에 집중하는 Transformer

#### 3-2. 사용자 행동을 순서(Sequence)로 접근

- 다음 행동은 장기적 평균 취향보다 최근 행동의 순서와 조합(문맥)에 더 큰 영향을 받음
- 예시 1 : 같은 아이템도 문맥에 따라 의미가 달라짐
    - 수영복→비치타월→선크림 : 선크림이 '휴가 준비물' → 방수팩, 래시가드 추천
    - 스킨→로션→선크림 : 선크림이 '데일리 스킨케어 단계' → 클렌징폼, 선크림 리필 추천
- 예시 2 : 시간 순서를 뛰어넘는 핵심 의도 (Long-Range Dependency)
    - 프린터→양말→라면→폰케이스→생수→샴푸→(?) : 관련 없는 행동 5개를 건너뛰고 프린터에 높은 가중치 → 토너 추천
    - RNN, GNN은 5단계를 거치며 정보가 희석되지만 Transformer는 직접 반영
- 두 예시 모두 현재 아이템과 다른 아이템 간의 관련성을 계산해 관련성 높은 정보를 더 많이 반영하는 방식 = Self-Attention

#### 3-3. Self-Attention

- 시퀀스 안의 모든 아이템 쌍의 관련성을 <span style="color:blue">거리와 상관없이 동시에</span> 계산
- 각 아이템 임베딩을 세 벡터로 변환
    - Query (Q) : 임베딩 × W_Q. 현재 아이템이 다른 아이템과의 관련성을 계산하는 기준
    - Key (K) : 임베딩 × W_K. Query와 내적되어 관련성 점수를 결정
    - Value (V) : 임베딩 × W_V. 관련성 점수에 따라 가중합되는 실제 정보
- $Attention(Q,K,V) = softmax\left(\dfrac{QK^T}{\sqrt d}\right) V$
- 계산 과정 (선크림 기준)
    - ① Q·K로 점수 계산 : 수영복 2.0 / 비치타월 1.5 / 양말 0.1
    - ② Softmax로 합이 1인 가중치로 변환 : 57.0% / 34.5% / 8.5%
    - ③ Value 가중합 : 0.570×V(수영복) + 0.345×V(비치타월) + 0.085×V(양말)
    - 수영복과 비치타월의 정보가 약 92% → '휴가 준비물'로 해석

#### 3-4. 주요 모델: SASRec vs BERT4Rec

- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> SASRec </span> : 단방향 어텐션(GPT와 유사)으로 과거 정보만 활용해 <span style="color:blue">다음 아이템을 예측</span>
    - 구조 : 아이템 임베딩 + 위치 임베딩 → Self-Attention(과거만 참조) → FFN → 출력과 아이템 임베딩을 내적해 다음 아이템 점수 계산
    - ex. [노트북→마우스→키보드] → 다음은 '노트북 거치대'
    - 특징 : 데이터가 희소하면 최근 아이템에, 풍부하면 긴 과거 맥락에 집중하도록 조절 / RNN과 달리 병렬 처리로 빠른 학습
    - 문제점 : Attention 연산량이 시퀀스 길이의 제곱(n²)에 비례 / 과거 방향만 보고, 장바구니 동시 담기처럼 순서가 엄격하지 않은 경우도 있음
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> BERT4Rec </span> : 양방향 어텐션(BERT와 유사)으로 과거와 미래를 모두 활용해 <span style="color:blue">가려진 아이템을 예측</span>
    - 시퀀스 일부를 [MASK]로 가림 → 가려진 위치가 앞뒤 모든 아이템을 참조
    - [MASK]로 가리는 이유 : 양방향 학습 시 target 아이템을 직접 보는 정보 누출(information leakage)을 막기 위해
    - ex. [노트북→마우스→[MASK]→키보드] → 앞뒤 문맥으로 '노트북 거치대' 예측
    - 특징 : 앞뒤 문맥을 모두 사용, 가리는 위치를 바꿔 여러 학습 샘플 생성

| 구분 | SASRec | BERT4Rec |
|:---:|:---:|:---:|
| 어텐션 방향 | 단방향 (과거만) | 양방향 (과거 + 미래) |
| 학습 목표 | 다음 아이템 예측 | 가려진 아이템 예측 |
| 학습 효율 | 빠름 | 상대적으로 느림 |

- <span style="color:green">BERT4Rec이 항상 더 좋은 것은 아님</span>
    - 원 논문(Sun et al., 2019) : BERT4Rec이 SASRec보다 높은 성능
    - 후속 연구(Klenitskiy & Vasilev, 2023) : 두 모델은 손실 함수가 달랐고, 맞추자 SASRec이 비슷하거나 더 높고 학습도 빨랐음
    - 성능 차이는 어텐션 방향보다 학습 설정의 영향이 컸음
- 상황별 선택
    - 상호작용이 많고 '다음 행동' 예측이 목적(실시간 피드, 커머스) → SASRec
    - 시퀀스가 짧은 경우 → 상황에 따라 다름 (BERT4Rec은 데이터 증강 효과가 있으나 손실 함수를 맞추면 차이가 줄어든다는 반론)
    - 행동 순서가 느슨한 도메인(장바구니 동시 담기) → BERT4Rec

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 4. 제언: 오늘날의 추천 시스템 </span>

#### 4-1. LLM의 시대

- 관점의 전환 : <span style="color:blue">추천을 랭킹 문제가 아닌 언어 문제로</span> 푼다
    - 과거 : (사용자 A, 아이템 123) → 0.8점
    - LLM : 점수가 아니라 문장으로 출력, ID가 아닌 의미(Semantic) 기반, 텍스트·이미지·오디오 활용(멀티모달), 추천 이유를 함께 제시하는 서술형 추천
- 새로운 가능성
    - 대화형 추천 : "비 오는 일요일에 혼자 보기 좋은, 너무 슬프지 않은 영화 추천해줘"
    - Zero-shot 추천 : 상호작용 데이터 없이 텍스트 설명만으로 추천 → 콜드 스타트의 강력한 해결책
- P5 (Geng et al., 2022) : 여러 추천 과제를 "텍스트 프롬프트 → 텍스트 답변"으로 통일해 하나의 언어 모델로 학습. 처음부터 학습해야 해서 자원과 시간이 많이 듦
- TALLRec (Bao et al., 2023) : 범용 LLM을 "좋아할까? (Yes/No)" 지시문으로 튜닝. LoRA로 일부 파라미터만 학습해 비용을 줄이고 적은 샘플로도 좋은 성능

#### 4-2. 인기 편향과 XAI

- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 인기 편향(Popularity Bias) </span> : 소수의 인기 아이템이 추천을 독점하고 대다수 아이템은 거의 추천되지 않는 현상
    - 강화 3단계 : 데이터 불균형 → 모델의 편향 강화 → Rich-get-richer (추천→노출→클릭→학습 데이터 반영 반복)
    - 사례 : Netflix Prize에서 인기 영화 16.5%가 전체 평점의 약 86% 차지 / Spotify 팟캐스트에서 개인화 시 스트리밍 28.90% 증가, 소비 다양성 11.51% 감소
- <span style="background-color:yellow; color:#000000; padding:0px; border-radius:5px;"> 설명가능성(XAI) </span> : "왜 추천했는가"를 설명할 수 있는가
    - GNN, Transformer 같은 딥러닝은 블랙박스 문제. 고객 설득과 비즈니스 담당자에게 근거 제시가 필요
    - <span style="color:green">LLM은 추천 이유를 자연어로 생성할 수 있지만, 그 설명이 실제 추천 근거와 일치한다는 보장은 없음</span>

#### 4-3. 컴퓨팅 비용과 Cold-Start를 고려한 도입

- 딥러닝 전면 도입은 비쌈 : GPU 비용, 실시간 스코어링 서버비, 대규모 트래픽의 지속 비용
- 현업(유튜브, 아마존 등)은 <span style="color:blue">2단계 구조</span>
    - 1단계 후보군 생성 : 수백만 개 → 수백 개. 선형 모델(내용 기반, 협업 필터링, 행렬 분해)로 빠르고 저렴하게
    - 2단계 순위 매기기 : 수백 개 후보를 정교하게 정렬해 상위 수십 개 노출. 딥러닝(GNN, Transformer, VAE)으로 비선형성, 순서, 관계 등 문맥을 총동원
- 결국 실제 추천 시스템은 단일 모델이 아니라 지금까지 배운 모델들을 단계별로 조합해 비용과 정확도를 함께 확보